# Yak fine-tuning on a mixture of datasets

Banking77, CLINC and SNLI are mixed into one training run, using the preprocessed files in `Data/processed/`
(`SftPreprocessing/Preprocessing.py`: one record per example with the text, or premise and hypothesis, and the label name).
Every training batch comes from one dataset (picked at random with the weights below), and inside a dataset the
candidate labels are re-drawn and re-shuffled every time an example is used. Candidate labels never cross datasets.

To test whether labels are *read* and not memorised, some classes are held out of training (their examples and
their names never appear). They are scored afterwards, together with all the other classes as candidates.

Helper modules (tokenizer wrapper, metrics, dataset mixing, inference) live in `Utils/` next to this notebook.
The packing code, the model, the architecture diagram and the evaluation helpers are defined in the cells below.

Run this notebook from anywhere inside the repository. Missing datasets are skipped.

## The whole pipeline on one page

```
 Data/processed/*.jsonl                 one record = text (or premise+hypothesis) + label NAME
        |
        v
 [mixture sampler]  pick a dataset for this batch (equal / sqrt / size weights)
        |
        v
 [label sampler]    pick N candidate labels (always incl. the true one), shuffle their order  (new draw every time)
        |
        v
 [packer]           [CLS] text [SEP] label1 [SEP] label2 [SEP] ... [SEP]      -> ids, label_pos
        |
        v
 [Yak encoder]      embed -> 4 x (attention + FFN) -> final LayerNorm              h : [B, L, 128]
        |
        v
 [gather]           h at label_pos (first token of each label)                     H : [B, M, 128]
        |
        v
 [scorer]           ONE shared Linear(128, 1) applied to every slot                scores : [B, M]
        |
        v
 [mask + softmax]   padded slots = -inf, softmax over the M slots                  p : [B, M]
        |
        v
 [loss]             cross-entropy against the slot that holds the true label
        |
        v
 [slot -> class]    map slots back to class ids, then accuracy / Brier / ECE / NLL
```

The only new weights are the scorer (128 weights + 1 bias). Everything about *which label is which* comes from the
label words the encoder reads, so a label the model never saw in training can still be scored.


In [17]:
import os, sys, json, random, time, types, subprocess

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:                                   # Google Colab: mount Drive and get the repository
    from google.colab import drive
    drive.mount("/content/drive")
    if not os.path.isdir("/content/Yak"):
        subprocess.run(["git", "clone", "--branch", "feature/finetune",
                        "https://github.com/gowthambalachandhiran/Yak.git", "/content/Yak"], check=True)
    os.chdir("/content/Yak")

def find_root(start="."):
    p = os.path.abspath(start)
    while True:
        if os.path.isdir(os.path.join(p, "Model")) and os.path.isdir(os.path.join(p, "Tokenizer")):
            return p
        if os.path.dirname(p) == p:
            raise RuntimeError("run this notebook inside the Yak repository (Model/ and Tokenizer/ not found)")
        p = os.path.dirname(p)

ROOT = find_root()
os.chdir(ROOT); sys.path.insert(0, ROOT)
print("repository root:", ROOT)

repository root: c:\Users\DELL\Downloads\Project Yak\Yak


In [18]:
# ---------------- configuration (edit these) ----------------
CKPT = "/content/drive/MyDrive/Yak/best.pt"   # best phase 1 checkpoint on Google Drive
# On a laptop with Drive for Desktop use its path instead, e.g. "/Users/<you>/Library/CloudStorage/GoogleDrive-<email>/My Drive/Yak/best.pt"
REQUIRE_CKPT = True                      # stop if the checkpoint is missing instead of silently training from random weights
OUT_DIR = "/content/drive/MyDrive/Yak/finetune_mixture" if IN_COLAB else "runs/mixture"   # best.pt and results.json go here
SOURCE = "processed"                     # "processed": Data/processed (preprocessing script output); "raw": Data/<name>/
PROCESSED_DIR = "Data/processed"        # train/validation/test.jsonl; in Colab point this at the folder on Drive
DATA_DIR = "Data"                        # only used when SOURCE = "raw"

TASKS = ["banking77", "clinc", "snli"]
HOLDOUT = {"banking77": 15, "clinc": 30, "snli": 0}      # classes kept out of training (to test unseen labels)
SNLI_TRAIN_LIMIT = None                                  # only for SOURCE = "raw" (549k pairs); processed data is already balanced
OOS_BOOST = 5                                            # repeat CLINC out-of-scope training rows this many times (1 = off)
MIX = "equal"                                            # "equal", "sqrt", "size", or a list like [1, 1, 2]
BATCH_SIZES = {"banking77": 32, "clinc": 16, "snli": 64} # CLINC batches are long (all 151 labels), so smaller

STEPS_PER_EPOCH = 1000
EPOCHS = 6
LR, HEAD_LR, WEIGHT_DECAY, WARMUP_FRAC = 2e-4, 1e-3, 0.01, 0.06
FREEZE_EPOCHS = 0
LOSS = "ce"                                              # "ce" or "brier"
FULL_PROB, MIN_LABELS = 0.5, 2                           # share of examples shown with all labels, minimum labels
EPOCH_EVAL_LIMIT = 1000                                  # examples per split in the per-epoch check (final eval uses all)
TTA = 4
SEED = 0
EVAL_BATCH = 64

D_MODEL, NHEAD, LAYERS, MAX_SEQ, DROPOUT = 128, 4, 4, 2048, 0.1   # must match the phase 1 checkpoint
os.makedirs(OUT_DIR, exist_ok=True)

## Packing text and labels
`[CLS] text [SEP] label1 [SEP] label2 [SEP] ...`, with the position of each label's first token recorded. Training re-draws the label subset and order every time an example is used.

In [19]:
# cell: packing
import random
from Utils.FastBpe import normalize_text, normalize_label


LABEL_ALIASES = {"oos": "out of scope"}   # names that carry no meaning as words


class Packer:
    def __init__(self, fast_bpe, class_names, max_labels=160, max_text_tokens=96,
                 max_label_tokens=12, aliases=None, max_premise_tokens=64, max_hypothesis_tokens=32):
        """
        fast_bpe:        FastBPE
        class_names:     list of class names, the position is the class id
        max_labels:      upper bound M on labels in one packed example
        max_text_tokens: the text is cut to this many tokens
        max_label_tokens: each label is cut to this many tokens
        """
        self.bpe = fast_bpe
        self.class_names = list(class_names)
        self.num_classes = len(self.class_names)
        self.max_labels = min(max_labels, self.num_classes)
        self.train_ids = list(range(self.num_classes))     # classes that may appear as training candidates
        self.max_text_tokens = max_text_tokens
        self.max_premise_tokens = max_premise_tokens
        self.max_hypothesis_tokens = max_hypothesis_tokens
        aliases = LABEL_ALIASES if aliases is None else aliases
        self.label_ids = []
        for name in self.class_names:
            ids = fast_bpe.encode_ids(normalize_label(aliases.get(name, name)), normalize=True)
            ids = ids[:max_label_tokens] or [fast_bpe.unk_id]
            self.label_ids.append(ids)

    # ------------------------------------------------------------------ text
    def encode_text(self, text):
        return self.bpe.encode_ids(text, normalize=True)[: self.max_text_tokens]

    def encode_pair(self, premise, hypothesis):
        """Sentence pair as one 'text': premise tokens [SEP] hypothesis tokens, each cut separately.
        The model has no segment embeddings, so the [SEP] between them is the only divider."""
        p = self.bpe.encode_ids(premise, normalize=True)[: self.max_premise_tokens]
        h = self.bpe.encode_ids(hypothesis, normalize=True)[: self.max_hypothesis_tokens]
        return p + [self.bpe.sep_id] + h

    def encode_row(self, row):
        if "premise" in row:
            return self.encode_pair(row["premise"], row["hypothesis"])
        return self.encode_text(row["text"])

    def max_text_len(self):
        return max(self.max_text_tokens, self.max_premise_tokens + 1 + self.max_hypothesis_tokens)

    # ------------------------------------------------------------------ packing
    def pack(self, text_ids, class_order):
        """Return (ids, label_pos) for the given text ids and classes in the given order."""
        ids = [self.bpe.cls_id] + list(text_ids) + [self.bpe.sep_id]
        label_pos = []
        for c in class_order:
            label_pos.append(len(ids))
            ids.extend(self.label_ids[c])
            ids.append(self.bpe.sep_id)
        return ids, label_pos

    def max_packed_len(self):
        longest = sorted((len(x) + 1 for x in self.label_ids), reverse=True)[: self.max_labels]
        return 2 + self.max_text_len() + sum(longest)

    def set_train_classes(self, class_ids):
        """Restrict training candidates to class_ids (the rest are held out and never shown in training).
        Evaluation still uses every class."""
        self.train_ids = sorted(class_ids)

    # ------------------------------------------------------------------ choosing the label list
    def train_labels(self, true_class, rng, full_prob=0.5, min_labels=2):
        """Random label list that always contains true_class.

        With probability full_prob use as many labels as allowed (so the model sees the real test
        condition), otherwise draw the count uniformly from min_labels..max_labels.
        """
        assert true_class in self.train_ids, f"class {true_class} is held out and cannot be a training target"
        cap = min(self.max_labels, len(self.train_ids))
        lo = min(min_labels, cap)
        n = cap if rng.random() < full_prob else rng.randint(lo, cap)
        others = [c for c in self.train_ids if c != true_class]
        chosen = rng.sample(others, n - 1) + [true_class]
        rng.shuffle(chosen)
        return chosen

    def eval_labels(self, true_class, rng=None):
        """All classes (up to max_labels). If there are more classes than max_labels, keep the true
        label plus a fixed-seed random subset. Order is class-id order unless rng shuffles it."""
        order = list(range(self.num_classes))
        if self.num_classes > self.max_labels:
            sub_rng = random.Random(true_class)
            others = [c for c in order if c != true_class]
            order = sorted(sub_rng.sample(others, self.max_labels - 1) + [true_class])
        if rng is not None:
            rng.shuffle(order)
        return order


class ClsDataset:
    """Map-style dataset (works with torch DataLoader, no torch import needed here).

    rows:   list of dicts with "text" and "label" (the jsonl format written by Data/*.py)
    train:  True  -> random label subsets and order, different every epoch (call set_epoch)
            False -> all labels in class order
    """

    def __init__(self, rows, packer, train, seed=0, full_prob=0.5, min_labels=2, eval_shuffle_seed=None, task_id=0):
        self.packer = packer
        self.task_id = task_id
        self.train = train
        self.seed = seed
        self.epoch = 0
        self.full_prob = full_prob
        self.min_labels = min_labels
        self.eval_shuffle_seed = eval_shuffle_seed   # None = class-id order; int = a fixed shuffled order
        self.text_ids = [packer.encode_row(r) for r in rows]      # single texts or premise/hypothesis pairs
        self.labels = [int(r["label"]) for r in rows]

    def set_epoch(self, epoch):
        self.epoch = epoch

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return self.example(idx, self.epoch * 10_007 + idx)

    def example(self, idx, nonce):
        """Example idx with a label list drawn from `nonce` (a different nonce gives a different list)."""
        true_class = self.labels[idx]
        if self.train:
            rng = random.Random(self.seed * 1_000_003 + nonce)
            order = self.packer.train_labels(true_class, rng, self.full_prob, self.min_labels)
        else:
            rng = None
            if self.eval_shuffle_seed is not None:
                rng = random.Random(self.eval_shuffle_seed * 1_000_003 + idx)
            order = self.packer.eval_labels(true_class, rng)
        ids, label_pos = self.packer.pack(self.text_ids[idx], order)
        return {"ids": ids, "label_pos": label_pos, "slot_class": order,
                "target": order.index(true_class), "true_class": true_class, "task_id": self.task_id}


def collate_lists(batch, pad_id):
    """Pad a list of examples to the longest sequence and the largest label count in the batch.

    Returns plain python lists so it can be tested without torch:
        ids [B, L] (pad_id), label_pos [B, M] (0 for unused), label_mask [B, M] (True = real label),
        slot_class [B, M] (-1 for unused), target [B], true_class [B]
    """
    L = max(len(b["ids"]) for b in batch)
    M = max(len(b["label_pos"]) for b in batch)
    out = {"ids": [], "label_pos": [], "label_mask": [], "slot_class": [], "target": [], "true_class": [],
           "task_id": []}
    for b in batch:
        n = len(b["label_pos"])
        out["ids"].append(b["ids"] + [pad_id] * (L - len(b["ids"])))
        out["label_pos"].append(b["label_pos"] + [0] * (M - n))
        out["label_mask"].append([True] * n + [False] * (M - n))
        out["slot_class"].append(b["slot_class"] + [-1] * (M - n))
        out["target"].append(b["target"])
        out["true_class"].append(b["true_class"])
        out["task_id"].append(b.get("task_id", 0))
    return out


def make_collate(pad_id):
    """Collate function for torch DataLoader: python lists -> tensors."""
    def collate(batch):
        import torch
        d = collate_lists(batch, pad_id)
        return {
            "ids": torch.tensor(d["ids"], dtype=torch.long),
            "label_pos": torch.tensor(d["label_pos"], dtype=torch.long),
            "label_mask": torch.tensor(d["label_mask"], dtype=torch.bool),
            "slot_class": torch.tensor(d["slot_class"], dtype=torch.long),
            "target": torch.tensor(d["target"], dtype=torch.long),
            "true_class": torch.tensor(d["true_class"], dtype=torch.long),
            "task_id": torch.tensor(d["task_id"], dtype=torch.long),
        }
    return collate


In [20]:
from Tokenizer.Tokenizer import BPE
from Utils import FastBPE
from Utils.Mixture import (load_task, load_processed, pick_holdout, apply_holdout, split_seen_unseen, mixture_weights,
                              MixtureBatchSampler, MixedTrainDataset)

bpe = BPE(); bpe.load("Tokenizer/TokenFile/bpe.json")
fb = FastBPE(bpe)
rng = random.Random(SEED)

if SOURCE == "processed":
    loaded = load_processed(PROCESSED_DIR, fb, Packer, datasets=TASKS, max_labels=160)
else:
    loaded = []
    for name in TASKS:
        t = load_task(name, DATA_DIR, fb, Packer, max_labels=160, seed=SEED)
        if t is None:
            print(f"skipping {name}: {DATA_DIR}/{name}/labels.json not found (run python Data/DownloadData.py --only {name})")
        else:
            loaded.append(t)

tasks = []
for t in loaded:
    name = t["name"]
    if OOS_BOOST > 1 and t["oos_index"] is not None:     # oos is ~0.7% of CLINC train but 18% of CLINC test
        oos_rows = [r for r in t["train"] if r["label"] == t["oos_index"]]
        t["train"] = t["train"] + oos_rows * (OOS_BOOST - 1)
        print(f"{name}: out-of-scope training rows {len(oos_rows)} -> {len(oos_rows) * OOS_BOOST} (x{OOS_BOOST})")
    if SNLI_TRAIN_LIMIT and name == "snli" and len(t["train"]) > SNLI_TRAIN_LIMIT:
        t["train"] = rng.sample(t["train"], SNLI_TRAIN_LIMIT)
    apply_holdout(t, pick_holdout(t, HOLDOUT.get(name, 0), seed=SEED))
    for split in ("val", "test"):
        seen, unseen = split_seen_unseen(t[split], t["held_out"])
        rng.shuffle(seen); rng.shuffle(unseen)
        t[split + "_seen"], t[split + "_unseen"] = seen, unseen
    tasks.append(t)

assert tasks, "no datasets found"
print(f"{'task':11s}{'classes':>8s}{'held out':>9s}{'train':>9s}{'val':>7s}{'test':>7s}{'chance':>8s}{'longest seq':>12s}")
for t in tasks:
    print(f"{t['name']:11s}{len(t['names']):8d}{len(t['held_out']):9d}{len(t['train']):9d}{len(t['val']):7d}"
          f"{len(t['test']):7d}{1 / len(t['names']):8.3f}{t['packer'].max_packed_len():12d}")
    if t["held_out"]:
        print("   held out:", ", ".join(t["names"][c] for c in t["held_out"][:8]), "..." if len(t["held_out"]) > 8 else "")

ImportError: cannot import name 'FastBPE' from 'Utils' (c:\Users\DELL\Downloads\Project Yak\Yak\Model\OurExperiments\FineTuning\Utils\__init__.py)

In [ ]:
mixed = MixedTrainDataset(tasks, ClsDataset, seed=SEED, full_prob=FULL_PROB, min_labels=MIN_LABELS)
sizes = mixed.sizes()
weights = mixture_weights(sizes, MIX)
batch_sizes = [BATCH_SIZES[t["name"]] for t in tasks]
sampler = MixtureBatchSampler(sizes, weights, batch_sizes, STEPS_PER_EPOCH, seed=SEED)

# how the mixture behaves over one epoch
n_batches = [0] * len(tasks)
for batch in sampler:
    n_batches[batch[0][0]] += 1
print(f"{'task':11s}{'weight':>8s}{'batches':>9s}{'examples seen':>15s}{'passes over data':>18s}")
for i, t in enumerate(tasks):
    seen = n_batches[i] * batch_sizes[i]
    print(f"{t['name']:11s}{weights[i]:8.2f}{n_batches[i]:9d}{seen:15d}{seen / sizes[i]:18.2f}")

# one decoded example per task
collate = make_collate(fb.pad_id)
for i, t in enumerate(tasks):
    ex = mixed[(i, 0, 1)]
    toks = " ".join(bpe.id2tok[x] for x in ex["ids"])
    print(f"\n[{t['name']}] {len(ex['slot_class'])} labels shown, true slot {ex['target']}")
    print(toks[:300] + (" ..." if len(toks) > 300 else ""))

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 3, figsize=(15, 3.6))
nm = [t["name"] for t in tasks]
ax[0].bar(nm, n_batches, color=["#4C78A8", "#F58518", "#54A24B"][:len(nm)]); ax[0].set_title("batches per epoch")
ax[1].bar(nm, [n_batches[i] * batch_sizes[i] / sizes[i] for i in range(len(nm))], color=["#4C78A8", "#F58518", "#54A24B"][:len(nm)])
ax[1].axhline(1, color="gray", ls=":"); ax[1].set_title("passes over each dataset per epoch (1 = once)")
ax[2].bar(nm, [len(t["names"]) for t in tasks], color=["#4C78A8", "#F58518", "#54A24B"][:len(nm)])
ax[2].set_title("classes per dataset (= largest M)")
for a in ax: a.set_axisbelow(True); a.grid(axis="y", alpha=0.3)
plt.tight_layout(); plt.show()

# how many labels does a batch really carry? (label counts drawn by the sampler)
fig, ax = plt.subplots(1, len(tasks), figsize=(5 * len(tasks), 3), squeeze=False)
for j, t in enumerate(tasks):
    ns = [len(mixed[(j, k % len(t["train"]), k)]["slot_class"]) for k in range(300)]
    ax[0, j].hist(ns, bins=20, color="#72B7B2"); ax[0, j].set_title(f"{t['name']}: labels shown per example")
    ax[0, j].set_xlabel("N")
plt.tight_layout(); plt.show()

### Look inside one packed example
The table lists every token of one real training example. `<-- label k` marks the first token of each candidate label:
those are the rows the model gathers and scores. Everything else is read by attention but never scored directly.

In [ ]:
ti = 0                                    # task index to show (0 = first task in TASKS)
ex = mixed[(ti, 0, 7)]                    # (task, row, nonce): a different nonce gives a different label draw / order
names = tasks[ti]["names"]
pos = {p: k for k, p in enumerate(ex["label_pos"])}
print(f"task {tasks[ti]['name']}: {len(ex['ids'])} tokens, {len(ex['slot_class'])} candidate labels, true slot = {ex['target']}\n")
print(f"{'idx':>4s}  {'token':<14s} note")
shown = 0
for i, tid in enumerate(ex["ids"]):
    note = ""
    if i in pos:
        k = pos[i]
        note = f"<-- label {k}: {names[ex['slot_class'][k]]}" + ("   (TRUE)" if k == ex["target"] else "")
    if note or i < 14 or i == len(ex["ids"]) - 1:
        print(f"{i:4d}  {bpe.id2tok[tid]:<14s} {note}")
        shown += 1
    elif shown and i == 14:
        print("   ...")
print("\nlabel_pos =", ex["label_pos"][:10], "..." if len(ex["label_pos"]) > 10 else "")
print("slot_class =", ex["slot_class"][:10], "..." if len(ex["slot_class"]) > 10 else "", " (slot -> class id)")

## Model
The phase 1 encoder with the MLM head removed and one shared `Linear(d_model, 1)` scorer.

Step-through with toy numbers (numpy only, no model needed)¶
One example with N = 3 real labels in M = 5 slots. Follow the colours left to right: gathered vectors -> one shared scorer -> mask padded slots -> softmax -> slot-to-class map. The scorer is the same 128 numbers for every slot; only the label words (through the encoder) make the slots differ.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rs = np.random.RandomState(3)
d, M, N = 8, 5, 3                       # d = 8 instead of 128 so it fits on screen
labels = ["refund", "card lost", "weather"]
slot_class = np.array([7, 2, 5, -1, -1])         # class ids in slot order; -1 = padding
H = rs.randn(M, d)                               # gathered vectors, one row per slot
w = rs.randn(d) * 0.6; b = 0.0                   # the shared scorer
scores = H @ w + b                               # [M]
mask = np.arange(M) < N
masked = np.where(mask, scores, -np.inf)
e = np.exp(masked - masked.max()); p = e / e.sum()
true_slot = 1
loss = -np.log(p[true_slot])

fig, ax = plt.subplots(1, 4, figsize=(16, 3.6), gridspec_kw={"width_ratios": [d, 1, 1, 1]})
ax[0].imshow(H, cmap="RdBu", aspect="auto"); ax[0].set_title("H: gathered [M, d]")
ax[0].set_yticks(range(M)); ax[0].set_yticklabels([f"slot {k}" + (f" ({labels[k]})" if k < N else " (pad)") for k in range(M)])
ax[1].imshow(scores[:, None], cmap="RdBu", aspect="auto"); ax[1].set_title("scores = H @ w + b")
ax[2].imshow(np.where(mask, scores, np.nan)[:, None], cmap="RdBu", aspect="auto"); ax[2].set_title("padded -> -inf")
ax[3].barh(range(M), p, color=["#54A24B" if k == true_slot else "#9ecae9" for k in range(M)]); ax[3].invert_yaxis()
ax[3].set_title(f"softmax p (true slot {true_slot}: loss {loss:.2f})")
for a in ax[1:3]: a.set_xticks([]); a.set_yticks(range(M))
for k in range(M):
    ax[1].text(0, k, f"{scores[k]:.2f}", ha="center", va="center")
    ax[2].text(0, k, f"{masked[k]:.2f}" if mask[k] else "-inf", ha="center", va="center")
    ax[3].text(p[k] + 0.01, k, f"{p[k]:.2f}", va="center")
plt.tight_layout(); plt.show()

print("slot -> class id:", slot_class.tolist())
class_probs = {int(c): round(float(p[k]), 3) for k, c in enumerate(slot_class) if c >= 0}
print("class probabilities:", class_probs, "| sum =", round(sum(class_probs.values()), 3))

# order invariance: shuffle the slots, scores follow the labels, class probabilities do not change
perm = np.array([2, 0, 1]); H2 = H[:N][perm]; s2 = H2 @ w + b
p2 = np.exp(s2 - s2.max()); p2 /= p2.sum()
print("after reordering labels", [labels[i] for i in perm], "->", {int(slot_class[i]): round(float(p2[k]), 3) for k, i in enumerate(perm)})

In [ ]:
# cell: classifier
import torch
import torch.nn as nn
import torch.nn.functional as F

from Model.LayerOne.EmbedPostional import EmbedPos
from Model.EncoderBlock import EncoderBlock


class YakClassifier(nn.Module):
    """Label-scoring classifier on top of the pre-trained Yak encoder.

    packed ids -> EmbedPos -> N x EncoderBlock -> final LayerNorm -> h [B, L, d]
    gather h at the first token of every label  -> H [B, M, d]
    shared scorer Linear(d, 1)                  -> scores [B, M]
    padded slots set to -inf                    -> softmax over slots happens in the loss / at inference

    The layer names (embed, blocks, final_norm) match MlmModel, so the phase 1 checkpoint loads directly.
    The MLM head and its bias are not part of this model.
    """

    def __init__(self, vocab_size: int, d_model: int, nhead: int, num_layers: int,
                 max_seq_len: int, dropout: float = 0.1, pad_token_id: int = 0):
        super().__init__()
        self.pad_id = pad_token_id
        self.embed = EmbedPos(vocab_size, d_model=d_model, max_len=max_seq_len,
                              pad_id=pad_token_id, dropout=dropout)
        self.blocks = nn.ModuleList([EncoderBlock(d_model, nhead, dropout) for _ in range(num_layers)])
        self.final_norm = nn.LayerNorm(d_model)
        self.scorer = nn.Linear(d_model, 1)           # the only new weights: d_model + 1
        nn.init.normal_(self.scorer.weight, mean=0.0, std=0.02)
        nn.init.zeros_(self.scorer.bias)

    # ------------------------------------------------------------------ weights
    def load_pretrained(self, path: str, map_location="cpu"):
        """Load embedding, blocks and final norm from a phase 1 checkpoint. Returns a short report."""
        try:
            sd = torch.load(path, map_location=map_location, weights_only=True)
        except Exception:                                   # checkpoint holds more than plain tensors
            sd = torch.load(path, map_location=map_location, weights_only=False)   # only for your own files
        meta = {k: v for k, v in sd.items() if isinstance(v, (int, float, str))} if isinstance(sd, dict) else {}
        if isinstance(sd, dict):
            for key in ("model", "model_state", "model_state_dict", "state_dict"):
                if key in sd and isinstance(sd[key], dict):
                    sd = sd[key]
                    break
        clean = {}
        for k, v in sd.items():
            for prefix in ("module.", "_orig_mod."):
                if k.startswith(prefix):
                    k = k[len(prefix):]
            if k.startswith(("embed.", "blocks.", "final_norm.")):
                clean[k] = v
        missing, unexpected = self.load_state_dict(clean, strict=False)
        missing = [k for k in missing if not k.startswith("scorer.")]   # scorer is new, expected missing
        if missing or unexpected or not clean:
            raise RuntimeError(f"checkpoint does not match: missing={missing[:5]} unexpected={unexpected[:5]} "
                               f"loaded={len(clean)}")
        return {"loaded_tensors": len(clean), "head_keys_ignored": [k for k in sd if k.startswith("head")],
                "checkpoint_info": meta}

    def num_parameters(self) -> int:
        return sum(p.numel() for p in self.parameters())

    def set_encoder_trainable(self, trainable: bool):
        """Freeze or unfreeze embedding and blocks. The final norm and scorer always train."""
        for p in self.embed.parameters():
            p.requires_grad = trainable
        for p in self.blocks.parameters():
            p.requires_grad = trainable

    # ------------------------------------------------------------------ forward
    def encode(self, ids: torch.Tensor) -> torch.Tensor:
        keep = ids != self.pad_id
        padding_mask = None if bool(keep.all()) else keep
        x = self.embed(ids)
        for block in self.blocks:
            x = block(x, padding_mask)
        return self.final_norm(x)

    def score(self, ids: torch.Tensor, label_pos: torch.Tensor, label_mask: torch.Tensor) -> torch.Tensor:
        """ids [B, L], label_pos [B, M] (0 for unused slots), label_mask [B, M] True for real labels.
        Returns scores [B, M] with -inf in unused slots."""
        h = self.encode(ids)                                                   # [B, L, d]
        idx = label_pos.unsqueeze(-1).expand(-1, -1, h.size(-1))               # [B, M, d]
        H = torch.gather(h, 1, idx)                                            # [B, M, d]
        scores = self.scorer(H).squeeze(-1)                                    # [B, M]
        return scores.masked_fill(~label_mask, float("-inf"))

    def forward(self, ids, label_pos, label_mask, target=None, loss_type: str = "ce"):
        """target is the slot index of the true label, shape [B]. Returns scores, or (loss, scores)."""
        scores = self.score(ids, label_pos, label_mask)
        if target is None:
            return scores
        return self.loss(scores, target, loss_type), scores

    @staticmethod
    def loss(scores: torch.Tensor, target: torch.Tensor, loss_type: str = "ce") -> torch.Tensor:
        scores = scores.float()
        if loss_type == "ce":
            return F.cross_entropy(scores, target)
        if loss_type == "brier":
            p = F.softmax(scores, dim=-1)                                      # -inf slots become exactly 0
            y = F.one_hot(target, p.size(-1)).to(p.dtype)
            return ((p - y) ** 2).sum(dim=-1).mean()
        raise ValueError(f"unknown loss {loss_type}")


In [ ]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(SEED)
model = YakClassifier(fb.vocab_size, D_MODEL, NHEAD, LAYERS, MAX_SEQ, DROPOUT, fb.pad_id)
if CKPT and os.path.exists(CKPT):
    print("loaded:", model.load_pretrained(CKPT))
elif REQUIRE_CKPT:
    folder = os.path.dirname(CKPT) or "."
    found = sorted(f for f in os.listdir(folder) if f.endswith(".pt")) if os.path.isdir(folder) else "folder not found"
    raise FileNotFoundError(f"checkpoint not found: {CKPT}\n.pt files in {folder}: {found}\n"
                            "Fix CKPT, or set REQUIRE_CKPT = False to train from random weights.")
else:
    print("WARNING: no checkpoint, training from random weights (the from-scratch baseline)")
model.to(device)
print(f"device {device} | parameters {model.num_parameters():,} (scorer adds {D_MODEL + 1})")

In [ ]:
# cell: diagram_text
def block_params(d_model, ff_multiplier=4):
    """Parameters of one pre-norm encoder block: fused qkv, output projection, two FFN layers, two LayerNorms."""
    d_ff = d_model * ff_multiplier
    attn = d_model * 3 * d_model + 3 * d_model + d_model * d_model + d_model
    ffn = d_model * d_ff + d_ff + d_ff * d_model + d_model
    norms = 2 * 2 * d_model
    return attn, ffn, norms


def expected_counts(vocab_size, d_model, num_layers):
    attn, ffn, norms = block_params(d_model)
    return {"embed": vocab_size * d_model, "blocks": [attn + ffn + norms] * num_layers,
            "final_norm": 2 * d_model, "scorer": d_model + 1}


def architecture_text(vocab_size, d_model, nhead, num_layers, counts, mlm_total=None, max_slots=None):
    """counts: {"embed": int, "blocks": [int, ...], "final_norm": int, "scorer": int}"""
    f = lambda n: f"{n:,}"
    blocks_total = sum(counts["blocks"])
    total = counts["embed"] + blocks_total + counts["final_norm"] + counts["scorer"]
    attn, ffn, norms = block_params(d_model)
    same = len(set(counts["blocks"])) == 1
    slots = f"M={max_slots}" if max_slots else "M"
    d, dh = d_model, d_model // nhead
    W, V = 64, 70                                   # box width, column of the label-position line

    def box(label="", num=""):                      # one line inside the encoder box, plus the side line
        return ("| " + label.ljust(W - 4 - 12) + num.rjust(12) + " |").ljust(V) + "|"

    def side(text=""):                              # a line outside the box with the side line still running
        return text.ljust(V) + "|"

    g = "  Gather the vector at each label's first token  <"
    gather = g + "-" * (V - len(g)) + "+   (0 parameters)"
    title = "+-- ENCODER, loaded from the phase 1 checkpoint "
    blocks_line = (f"per block {f(counts['blocks'][0])}, x {num_layers}" if same else "blocks")
    L = [
        ("packed ids [B, L]").ljust(V) + f"label positions [B, {slots}]",
        side("        |"),
        side("        v"),
        (title + "-" * (W - len(title) - 1) + "+").ljust(V) + "|",
        box("Embedding + sinusoidal positions"),
        box(f"  [B, L] -> [B, L, {d}]", f(counts["embed"])),
        box(f"  {vocab_size:,} x {d} table, positions add 0"),
        box(),
        box(f"Encoder block x {num_layers}   [B, L, {d}] -> [B, L, {d}]"),
        box(f"  attention, {nhead} heads of {dh}, bidirectional", f(attn)),
        box(f"  FFN {d} -> {4 * d} -> {d}", f(ffn)),
        box("  2 LayerNorms", f(norms)),
        box(f"  {blocks_line}", f(blocks_total)),
        box(),
        box("Final LayerNorm", f(counts["final_norm"])),
        ("+" + "-" * (W - 2) + "+").ljust(V) + "|",
        side(f"      | h  [B, L, {d}]"),
        side("      v"),
        gather,
        f"      [B, L, {d}] -> [B, {slots}, {d}]",
        "      |",
        "      v",
        f"  SHARED SCORER  Linear({d} -> 1)  (NEW)      {f(counts['scorer']):>9s} parameters",
        f"      [B, {slots}, {d}] -> [B, {slots}], same weights for every label",
        "      |",
        "      v   unused slots set to -inf",
        "  Softmax over the label slots",
        "      |",
        "      v",
        "  Loss (cross-entropy or Brier) against the correct slot",
        "",
        "  REMOVED from phase 1:",
        f"    MLM head  Linear({d} -> {vocab_size:,}), tied to the embedding (no weights of its own)",
        f"    head bias, {f(vocab_size)} parameters",
        "",
        f"  Parameters: {f(counts['embed'])} + {f(blocks_total)} + {f(counts['final_norm'])} + {f(counts['scorer'])} = {f(total)}",
    ]
    if mlm_total is not None:
        L.append(f"  Phase 1 model: {f(mlm_total)}  ->  {f(mlm_total)} - {f(vocab_size)} (head bias) + "
                 f"{f(counts['scorer'])} (scorer) = {f(mlm_total - vocab_size + counts['scorer'])}")
    return "\n".join(L)


In [ ]:
from Model.MlmModel import MlmModel

def n_params(m):
    return sum(p.numel() for p in m.parameters())

counts = {"embed": n_params(model.embed), "blocks": [n_params(b) for b in model.blocks],
          "final_norm": n_params(model.final_norm), "scorer": n_params(model.scorer)}
phase1 = MlmModel(fb.vocab_size, D_MODEL, NHEAD, LAYERS, MAX_SEQ, DROPOUT, fb.pad_id)   # for the before/after count
phase1_total = phase1.num_parameters(); del phase1

print(architecture_text(fb.vocab_size, D_MODEL, NHEAD, LAYERS, counts, phase1_total, max_slots=max(len(t["names"]) for t in tasks)))
print()
print("matches the formula for this configuration:", counts == expected_counts(fb.vocab_size, D_MODEL, LAYERS))
print("model has no MLM head:", not any(k.startswith("head") for k in model.state_dict()))

## Training and evaluation helpers

In [ ]:
# cell: eval_helpers
import contextlib
import numpy as np
import torch
from torch.utils.data import DataLoader
from Model.OurExperiments.Finetuning.Utils.Metrics import softmax, slots_to_class_probs

def amp_setup(device, use_amp):
    if use_amp and device.type == "cuda":
        # bf16 needs an Ampere (compute capability 8+) GPU; a T4 is 7.5 and should use fp16 (plus loss scaling)
        dtype = torch.bfloat16 if torch.cuda.get_device_capability(0)[0] >= 8 else torch.float16
        return (lambda: torch.autocast("cuda", dtype=dtype)), dtype
    return (lambda: contextlib.nullcontext()), None


@torch.no_grad()
def predict(model, dataset, device, batch_size, collate, autocast):
    """Return slot scores [N, M], slot classes [N, M] (-1 = unused), target slots [N], true classes [N]."""
    model.eval()
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False, collate_fn=collate)
    S, C, T, Y = [], [], [], []
    for b in loader:
        with autocast():
            scores = model(b["ids"].to(device), b["label_pos"].to(device), b["label_mask"].to(device))
        S.append(scores.float().cpu().numpy())
        C.append(b["slot_class"].numpy())
        T.append(b["target"].numpy())
        Y.append(b["true_class"].numpy())
    M = max(s.shape[1] for s in S)
    S = np.concatenate([np.pad(s, ((0, 0), (0, M - s.shape[1])), constant_values=-np.inf) for s in S])
    C = np.concatenate([np.pad(c, ((0, 0), (0, M - c.shape[1])), constant_values=-1) for c in C])
    return S, C, np.concatenate(T), np.concatenate(Y)


def eval_probs(model, rows, packer, args, device, collate, autocast, n_orders=1, temperature=1.0):
    """Class probabilities [N, K] averaged over n_orders label orders (the first is class-id order)."""
    acc, true, first = None, None, None
    for k in range(n_orders):
        ds = ClsDataset(rows, packer, train=False, eval_shuffle_seed=None if k == 0 else k)
        S, C, T, Y = predict(model, ds, device, args.eval_batch_size, collate, autocast)
        if k == 0:
            first = (S, T)
        P = slots_to_class_probs(softmax(S, temperature), C, packer.num_classes)
        acc = P if acc is None else acc + P
        true = Y
    return acc / n_orders, true, first


In [ ]:
from torch.utils.data import DataLoader
from Model.OurExperiments.Finetuning.Utils.Metrics import report, fit_temperature, oos_report

ev_args = types.SimpleNamespace(eval_batch_size=EVAL_BATCH)
autocast, amp_dtype = amp_setup(device, True)
scaler = torch.cuda.amp.GradScaler(enabled=(amp_dtype == torch.float16))

head_params = list(model.scorer.parameters()) + list(model.final_norm.parameters())
head_ids = {id(p) for p in head_params}
enc_decay, enc_nodecay = [], []
for n, p in model.named_parameters():
    if id(p) in head_ids:
        continue
    (enc_nodecay if (p.ndim < 2 or "embed" in n) else enc_decay).append(p)
opt = torch.optim.AdamW([
    {"params": enc_decay, "lr": LR, "weight_decay": WEIGHT_DECAY},
    {"params": enc_nodecay, "lr": LR, "weight_decay": 0.0},
    {"params": head_params, "lr": HEAD_LR, "weight_decay": 0.0},
])
loader = DataLoader(mixed, batch_sampler=sampler, collate_fn=collate, num_workers=0)
total_steps = EPOCHS * STEPS_PER_EPOCH
warm = max(1, int(WARMUP_FRAC * total_steps))
sched = torch.optim.lr_scheduler.LambdaLR(
    opt, lambda s: (s + 1) / warm if s < warm else max(0.0, (total_steps - s) / max(1, total_steps - warm)))

def task_eval(t, split, n_orders=1, temp=1.0, limit=None):
    rows = t[split]
    rows = rows if limit is None else rows[:limit]
    if not rows:
        return None
    P, Y, first = eval_probs(model, rows, t["packer"], ev_args, device, collate, autocast, n_orders, temp)
    return report(P, Y), P, Y, first

In [ ]:
history, loss_curves, best = [], {t["name"]: [] for t in tasks}, None
running = {t["name"]: None for t in tasks}
best_path = os.path.join(OUT_DIR, "best.pt")

for ep in range(EPOCHS):
    sampler.set_epoch(ep)
    model.set_encoder_trainable(ep >= FREEZE_EPOCHS)
    model.train()
    t0 = time.time()
    for i, b in enumerate(loader):
        name = tasks[int(b["task_id"][0])]["name"]
        b = {k: v.to(device) for k, v in b.items()}
        with autocast():
            loss, _ = model(b["ids"], b["label_pos"], b["label_mask"], b["target"], LOSS)
        opt.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        scaler.unscale_(opt)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(opt); scaler.update(); sched.step()
        r = running[name]
        running[name] = loss.item() if r is None else 0.95 * r + 0.05 * loss.item()
        loss_curves[name].append(running[name])
        if (i + 1) % 250 == 0:
            print(f"  epoch {ep} step {i + 1}/{STEPS_PER_EPOCH} " + " ".join(f"{k} {v:.3f}" for k, v in running.items() if v is not None))
    row = {"epoch": ep}
    for t in tasks:
        for kind in ("val_seen", "val_unseen"):
            out = task_eval(t, kind, limit=EPOCH_EVAL_LIMIT)
            row[f"{t['name']}_{kind}_acc"] = None if out is None else out[0]["acc"]
            if kind == "val_seen" and t["oos_index"] is not None:
                o = oos_report(out[1], out[2], t["oos_index"])
                row[f"{t['name']}_in_scope_acc"], row[f"{t['name']}_oos_recall"] = o["in_scope_acc"], o["oos_recall"]
    history.append(row)
    print(f"epoch {ep} ({time.time() - t0:.0f}s) val accuracy, seen / unseen classes:")
    for t in tasks:
        s, u = row[f"{t['name']}_val_seen_acc"], row[f"{t['name']}_val_unseen_acc"]
        print(f"   {t['name']:11s} {s:.3f}" + (f" / {u:.3f}" if u is not None else ""), end="")
        if t["oos_index"] is not None:
            print(f"   in-scope {row[t['name'] + '_in_scope_acc']:.3f}  out-of-scope recall {row[t['name'] + '_oos_recall']:.3f}", end="")
        print()
    score = sum(row[f"{t['name']}_val_seen_acc"] for t in tasks) / len(tasks)
    if best is None or score > best:
        best = score
        torch.save({"model": model.state_dict(), "tasks": [t["name"] for t in tasks],
                    "held_out": {t["name"]: t["held_out"] for t in tasks}, "epoch": ep}, best_path)
print("best mean seen-class val accuracy:", round(best, 4), "->", best_path)

In [ ]:
model.load_state_dict(torch.load(best_path, map_location=device)["model"])

final = {}
for t in tasks:
    _, _, _, first = task_eval(t, "val_seen")
    t["temperature"] = fit_temperature(first[0], first[1])          # fitted on seen-class validation only
    print(f"\n{t['name']}  (temperature {t['temperature']:.2f}, chance {1 / len(t['names']):.3f})")
    print(f"  {'classes':9s}{'setting':16s}{'n':>6s}{'acc':>8s}{'brier':>8s}{'ece':>8s}{'nll':>8s}")
    for kind in ("test_seen", "test_unseen"):
        for tag, n_orders, temp in (("single order", 1, 1.0), (f"{TTA} orders", TTA, 1.0), (f"{TTA} orders + T", TTA, t["temperature"])):
            out = task_eval(t, kind, n_orders, temp)
            if out is None:
                continue
            r = out[0]; final[(t["name"], kind, tag)] = r
            print(f"  {kind[5:]:9s}{tag:16s}{len(t[kind]):6d}{r['acc']:8.4f}{r['brier']:8.4f}{r['ece']:8.4f}{r['nll']:8.4f}")
            if t["oos_index"] is not None and kind == "test_seen" and tag == f"{TTA} orders + T":
                o = oos_report(out[1], out[2], t["oos_index"])
                final[(t["name"], "oos_split", tag)] = o
                print(f"    in-scope accuracy {o['in_scope_acc']:.4f} | out-of-scope recall {o['oos_recall']:.4f}, "
                      f"precision {o['oos_precision']:.4f} | real intents flagged out-of-scope {o['in_scope_flagged_oos']:.4f} "
                      f"| out-of-scope share of rows {o['oos_share']:.3f}")
json.dump({"history": history, "temperatures": {t["name"]: t["temperature"] for t in tasks},
           "held_out": {t["name"]: [t["names"][c] for c in t["held_out"]] for t in tasks},
           "final": {"|".join(k): v for k, v in final.items()}},
          open(os.path.join(OUT_DIR, "results.json"), "w"), indent=2)

## Seen vs held-out classes, side by side

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
tag = f"{TTA} orders + T"
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
x = np.arange(len(tasks)); wdt = 0.35
seen = [final.get((t["name"], "test_seen", tag), {}).get("acc", 0) for t in tasks]
unseen = [final.get((t["name"], "test_unseen", tag), {}).get("acc", 0) for t in tasks]
chance = [1 / len(t["names"]) for t in tasks]
ax[0].bar(x - wdt / 2, seen, wdt, label="seen classes"); ax[0].bar(x + wdt / 2, unseen, wdt, label="held-out classes")
ax[0].scatter(x, chance, color="k", marker="_", s=900, label="chance")
ax[0].set_xticks(x); ax[0].set_xticklabels([t["name"] for t in tasks]); ax[0].set_ylim(0, 1.05)
ax[0].set_title("test accuracy"); ax[0].legend()
for i, (s, u) in enumerate(zip(seen, unseen)):
    ax[0].text(i - wdt / 2, s + 0.01, f"{s:.2f}", ha="center"); ax[0].text(i + wdt / 2, u + 0.01, f"{u:.2f}", ha="center")
labs = ["single order", f"{TTA} orders", tag]
for k, lab in enumerate(labs):
    vals = [final.get((t["name"], "test_seen", lab), {}).get("ece", 0) for t in tasks]
    ax[1].bar(x + (k - 1) * 0.27, vals, 0.27, label=lab)
ax[1].set_xticks(x); ax[1].set_xticklabels([t["name"] for t in tasks]); ax[1].set_title("ECE on seen test classes (lower is better)"); ax[1].legend()
plt.tight_layout(); plt.show()

for t in tasks:
    o = final.get((t["name"], "oos_split", tag))
    if o:
        print(f"{t['name']} out-of-scope: recall {o['oos_recall']:.3f} | precision {o['oos_precision']:.3f} | in-scope accuracy {o['in_scope_acc']:.3f}")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

n = len(tasks)
fig, ax = plt.subplots(2, n, figsize=(5 * n, 8), squeeze=False)
for j, t in enumerate(tasks):
    ax[0, j].plot(loss_curves[t["name"]]); ax[0, j].set_title(f"{t['name']}: training loss"); ax[0, j].set_xlabel("batches of this task")
    ax[1, j].plot([h[f"{t['name']}_val_seen_acc"] for h in history], marker="o", label="seen classes")
    if t["held_out"]:
        ax[1, j].plot([h[f"{t['name']}_val_unseen_acc"] for h in history], marker="o", label="held-out classes")
    ax[1, j].axhline(1 / len(t["names"]), color="gray", ls=":", label="chance")
    ax[1, j].set_title(f"{t['name']}: validation accuracy"); ax[1, j].set_xlabel("epoch"); ax[1, j].legend()
plt.tight_layout(); plt.show()

fig, ax = plt.subplots(1, n, figsize=(5 * n, 4), squeeze=False)
for j, t in enumerate(tasks):
    for tag, style in (("single order", "--"), (f"{TTA} orders + T", "-")):
        temp = 1.0 if tag == "single order" else t["temperature"]
        out = task_eval(t, "test_seen", 1 if tag == "single order" else TTA, temp)
        _, P, Y, _ = out
        conf, hit = P.max(1), (P.argmax(1) == Y)
        edges = np.linspace(0, 1, 11); xs, ys = [], []
        for lo, hi in zip(edges[:-1], edges[1:]):
            m = (conf > lo) & (conf <= hi)
            if m.sum() >= 20:
                xs.append(conf[m].mean()); ys.append(hit[m].mean())
        ax[0, j].plot(xs, ys, style, marker="o", label=tag)
    ax[0, j].plot([0, 1], [0, 1], color="gray", ls=":"); ax[0, j].set_title(f"{t['name']}: reliability (test, seen)")
    ax[0, j].set_xlabel("confidence"); ax[0, j].set_ylabel("accuracy"); ax[0, j].legend()
plt.tight_layout(); plt.show()

In [ ]:
from Utils.Infer import classify

demos = [
    ("my card has not arrived after two weeks", ["card delivery delay", "forgot password", "exchange rate question", "refund request"]),
    ("how do i change my pin", ["change pin", "card arrival", "weather forecast", "play music"]),
    ("what is the weather like tomorrow", ["weather", "book a flight", "transfer money", "tell a joke"]),
    (("A man plays a guitar on a stage.", "A man is making music."), ["true", "maybe", "false"]),
]
for text, labels in demos:
    out = classify(model, fb, text, labels, Packer, device=device, n_orders=4)
    print(text)
    print("   ", [(l, round(p, 2)) for l, p in out], "\n")

How to read the results:
- **Seen vs held-out classes.** Held-out accuracy well above chance (1 / number of classes) means the model matches a
  label's wording to the text. Held-out accuracy near chance means it only memorised the training labels.
- **Training sees fewer labels than the test does.** With classes held out, training candidates come from the seen
  classes only (for example 62 of 77), while evaluation shows all of them. That shift is part of the test.
- **Temperature** is fitted on seen-class validation data per dataset and applied to both seen and held-out test rows.
- **Mixture weights.** `equal` shows each dataset a third of the batches. Check the "passes over data" column: a value
  far above 1 means that dataset repeats a lot and can overfit, so lower its weight or add data.
- The demo cell uses label wordings the model has never seen, so expect weak results unless held-out accuracy is good.